In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('missing_treated.csv')
print(df.columns.tolist())

['country', 'date', 'agricultural_land%', 'forest_land%', 'land_area', 'avg_precipitation', 'trade_in_services%', 'control_of_corruption_estimate', 'control_of_corruption_std', 'access_to_electricity%', 'renewvable_energy_consumption%', 'electric_power_consumption', 'co2_emisions', 'other_greenhouse_emisions', 'inflation_annual%', 'real_interest_rate', 'research_and_development_expenditure%', 'central_goverment_debt%', 'tax_revenue%', 'expense%', 'goverment_effectiveness_estimate', 'goverment_effectiveness_std', 'individuals_using_internet%', 'military_expenditure%', 'gdp_current_us', 'political_stability_estimate', 'political_stability_std', 'rule_of_law_estimate', 'rule_of_law_std', 'regulatory_quality_estimate', 'regulatory_quality_std', 'government_expenditure_on_education%', 'government_health_expenditure%', 'multidimensional_poverty_headcount_ratio%', 'gini_index', 'birth_rate', 'death_rate', 'life_expectancy_at_birth', 'population', 'rural_population', 'voice_and_accountability_

In [ ]:
print(df.shape)

(10721, 58)


In [ ]:
class1_cols = [
    'access_to_electricity%',
    'individuals_using_internet%',
    'renewvable_energy_consumption%',
    'government_expenditure_on_education%',
    'government_health_expenditure%',
    'school enrollment, primary (% net)',
    'school enrollment, secondary (% net)',
    'literacy rate, adult total (% of people ages 15 and above)',
    'literacy rate, youth (ages 15-24), gender parity index (gpi)',
    'people using at least basic drinking water services (% of population)',
    'people using at least basic sanitation services (% of population)'
]


In [ ]:
def min_max_normalize(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5  # neutral

    return (series - min_val) / (max_val - min_val)

df[class1_cols] = df.groupby('date')[class1_cols].transform(min_max_normalize)

In [ ]:
# List of Class 2 columns
class2_cols = [
    'co2_emisions',
    'other_greenhouse_emisions',
    'inflation_annual%',
    'birth_rate',
    'death_rate',
    'mortality rate, infant (per 1,000 live births)',
    'mortality rate, under-5 (per 1,000 live births)',
    'intentional_homicides',
    'multidimensional_poverty_headcount_ratio%',
    'unemployment with basic education (% of total labor force with basic education)'
]

# Reverse Min-Max normalization function
def reverse_min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5  # neutral

    return (max_val - series) / (max_val - min_val)

# Apply year-wise normalization
df[class2_cols] = df.groupby('date')[class2_cols].transform(reverse_min_max)

In [ ]:
# Class 3 columns
class3_cols = [
    'gdp_current_us',
    'population',
    'electric_power_consumption',
    'population_density'
]

# Replace 0 with NaN (important)
df[class3_cols] = df[class3_cols].replace(0, np.nan)

# Log transform
df_log = df[class3_cols].apply(lambda x: np.log(x))

# Min-Max normalization (year-wise)
def min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (series - min_val) / (max_val - min_val)

df[class3_cols] = df_log.groupby(df['date']).transform(min_max)

In [ ]:
# Class 4 columns
class4_cols = [
    'control_of_corruption_estimate',
    'goverment_effectiveness_estimate',
    'political_stability_estimate',
    'rule_of_law_estimate',
    'regulatory_quality_estimate',
    'voice_and_accountability_estimate'
]

# Normalize from [-2.5, 2.5] → [0, 1]
df[class4_cols] = df[class4_cols].apply(lambda x: (x + 2.5) / 5)

In [ ]:
# Positive variables
class5_pos = [
    'trade_in_services%',
    'exports of goods and services (% of gdp)',
    'tax_revenue%',
    'research_and_development_expenditure%'
]

# Negative variables
class5_neg = [
    'imports of goods and services (% of gdp)',
    'central_goverment_debt%',
    'military_expenditure%',
    'gini_index',
    'expense%'
]

# Min-max function
def min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (series - min_val) / (max_val - min_val)

# Reverse min-max
def reverse_min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (max_val - series) / (max_val - min_val)

# Apply normalization (year-wise)
df[class5_pos] = df.groupby('date')[class5_pos].transform(min_max)
df[class5_neg] = df.groupby('date')[class5_neg].transform(reverse_min_max)

In [ ]:
# Class 6 columns
class6_cols = [
    'life_expectancy_at_birth',
    'labor force participation rate for ages 15-24, total (%) (national estimate)',
    'school enrollment, tertiary (gross), gender parity index (gpi)'
]

# Min-max normalization function
def min_max(series):
    min_val = series.min()
    max_val = series.max()

    if pd.isna(min_val) or pd.isna(max_val) or max_val == min_val:
        return series * 0 + 0.5

    return (series - min_val) / (max_val - min_val)

# Apply year-wise normalization
df[class6_cols] = df.groupby('date')[class6_cols].transform(min_max)

In [ ]:
# Split handling
class7_pos = ['agricultural_land%', 'forest_land%']
class7_neg = ['rural_population']

df[class7_pos] = df.groupby('date')[class7_pos].transform(min_max)
df[class7_neg] = df.groupby('date')[class7_neg].transform(reverse_min_max)

In [ ]:
df.head()

,country,date,agricultural_land%,forest_land%,land_area,avg_precipitation,trade_in_services%,control_of_corruption_estimate,control_of_corruption_std,access_to_electricity%,...,"mortality rate, infant (per 1,000 live births)","mortality rate, under-5 (per 1,000 live births)",people using at least basic drinking water services (% of population),people using at least basic sanitation services (% of population),"school enrollment, primary (% net)","school enrollment, secondary (% net)","school enrollment, tertiary (gross), gender parity index (gpi)",unemployment with basic education (% of total labor force with basic education),homicide_available,population_density
0,AFGHANISTAN,1960-01-01,NaN,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,0.0,0.024978,NaN,NaN,NaN,NaN,NaN,NaN,1,0.424077
1,AFGHANISTAN,1961-01-01,0.633862,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,0.0,0.194225,NaN,NaN,NaN,NaN,NaN,NaN,1,0.423754
2,AFGHANISTAN,1962-01-01,0.634711,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,0.0,0.207101,NaN,NaN,NaN,NaN,NaN,NaN,1,0.423279
3,AFGHANISTAN,1963-01-01,0.635561,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,0.0,0.220389,NaN,NaN,NaN,NaN,NaN,NaN,1,0.422568
4,AFGHANISTAN,1964-01-01,0.636505,NaN,652230.0,327.0,NaN,NaN,NaN,NaN,...,0.0,0.218437,NaN,NaN,NaN,NaN,NaN,NaN,1,0.422274


In [ ]:
print(max(df['country']))

MONGOLIA


In [ ]:
df.to_csv('pre-processed dataset.csv')

from google.colab import files
files.download('pre-processed dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>